# Analyse des votes de vecteurs par TYPE de patch

**But** : pour une texture et un type de patch choisis, montrer comment les vecteurs internes de chaque patch votent (crop + carte spatiale + histogramme des votes).

**4 types** :
- **Type 1 — Unanime cohérent** : accord élevé ET vote == étiquette
- **Type 2 — Divisé en zones** : accord bas, 2 textures dominantes
- **Type 3 — Éparpillé** : accord bas, ≥3 textures mélangées
- **Type 4 — Mal classé** : vote_majoritaire ≠ étiquette

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELLULE 1 — CONFIG  (paramètres à modifier ici)
# ══════════════════════════════════════════════════════════════════════
import sys, os, pickle, logging, tempfile, zipfile
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image as PILImage
import h5py
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-7s %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("vote_analysis")

# ── Chemins ──────────────────────────────────────────────────────────
ROOT      = Path("/home/aidouni/meb_texture_seg")
H5_PATH   = ROOT / "data/feature_database/database_meb_ouassim.h5"
IMG_DIR   = ROOT / "Image_Ouassim"
CACHE_DIR = ROOT / "vote_analysis_cache"
CKPT_PATH = ROOT / "checkpoints/sam2.1_hiera_small_1.pt"
CKPT_DIR  = ROOT / "checkpoints/sam2.1_hiera_small_1"

# ── Blocs : un par stage ─────────────────────────────────────────────
# stage1→block_0 (~1075 vec/patch)  stage2→block_2 (~252)
# stage3→block_7 (~60)              stage4→block_15 (~15)
BLOCS = ["block_0", "block_2", "block_7", "block_15"]
STAGE_LABELS = {
    "block_0":  "Stage 1 (block_0)",
    "block_2":  "Stage 2 (block_2)",
    "block_7":  "Stage 3 (block_7)",
    "block_15": "Stage 4 (block_15)",
}

# ── Textures ─────────────────────────────────────────────────────────
TEXTURES = [1, 3, 4, 5, 6, 7, 9]
TNAMES = {
    1: "Tot.homogène", 3: "Faisceaux",  4: "Filaments",
    5: "Strat.rect",  6: "Strat.sin",  7: "Granuleux",  9: "Trou",
}
TEX_COLORS = {
    1: "#2ecc71", 3: "#3498db", 4: "#e74c3c",
    5: "#9b59b6", 6: "#f39c12", 7: "#1abc9c",
    9: "#e67e22", -1: "#cccccc",
}

# ── Hyperparamètres ───────────────────────────────────────────────────
PCA_DIM  = 50
LP_C     = 1.0
SEED     = 42
IMG_SIZE = 1024

# ── Affichage — cellules 6 & 7 ───────────────────────────────────────
# None  = afficher TOUS les vecteurs dans la carte spatiale (plus riche).
# entier = sous-échantillonner à N pour l'affichage (plus rapide à tracer).
# Le vote (classification) utilise TOUJOURS tous les vecteurs.
N_VEC_AFFICHAGE = None

# ── Seuils de classification (relancer cellule 5 après modif) ────────
SEUIL_UNANIME = 0.75   # accord >= seuil → unanime (type 1 ou 4)
SEUIL_2ZONES  = 0.80   # top-2 textures cumulent >= seuil → divisé (type 2)

# ── Forcer recalcul (mettre True puis relancer les cellules lourdes) ─
FORCE_RECOMPUTE = False

_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(3, 1, 1)
_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(3, 1, 1)

CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"ROOT      : {ROOT}")
print(f"H5_PATH   : {H5_PATH}  →  existe={H5_PATH.exists()}")
print(f"IMG_DIR   : {IMG_DIR}  →  existe={IMG_DIR.exists()}")
print(f"CACHE_DIR : {CACHE_DIR}")
print(f"BLOCS     : {BLOCS}")
print(f"N_VEC_AFFICHAGE={N_VEC_AFFICHAGE}  (None = tous les vecteurs en affichage)")
print(f"SEUIL_UNANIME={SEUIL_UNANIME}   SEUIL_2ZONES={SEUIL_2ZONES}")
print(f"FORCE_RECOMPUTE={FORCE_RECOMPUTE}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELLULE 3 — EXTRACTION VECTEURS LOCAUX  (lourd — exécuté UNE fois)
# TOUS les vecteurs de chaque région sont extraits (pas de cap).
# Vote et classification portent sur l'ensemble complet des vecteurs.
# Si cache présent pour tous les blocs → rechargement instantané.
# Pour forcer le recalcul : mettre FORCE_RECOMPUTE=True en cellule 1.
# ══════════════════════════════════════════════════════════════════════
import torch

sys.path.insert(0, str(ROOT / "TextureSAM" / "sam2"))
from sam2.modeling.backbones.hieradet import Hiera
from sam2.modeling.backbones.image_encoder import ImageEncoder, FpnNeck
from sam2.modeling.position_encoding import PositionEmbeddingSine

# ── Utilitaires modèle ────────────────────────────────────────────────

def _build_encoder():
    trunk = Hiera(
        embed_dim=96, num_heads=1, stages=(1, 2, 11, 2),
        global_att_blocks=(7, 10, 13),
        window_pos_embed_bkg_spatial_size=(7, 7),
    )
    neck = FpnNeck(
        position_encoding=PositionEmbeddingSine(
            num_pos_feats=256, normalize=True, scale=None, temperature=10000),
        d_model=256, backbone_channel_list=[768, 384, 192, 96],
        kernel_size=1, stride=1, padding=0,
        fpn_interp_model="nearest", fuse_type="sum", fpn_top_down_levels=[2, 3],
    )
    return ImageEncoder(trunk=trunk, neck=neck, scalp=1)


def _load_model(device):
    enc = _build_encoder()
    tmp = None
    if CKPT_PATH.is_file():
        sd = torch.load(CKPT_PATH, map_location="cpu", weights_only=True)
    elif CKPT_DIR.is_dir():
        arch = CKPT_DIR / "archive" if (CKPT_DIR / "archive").is_dir() else CKPT_DIR
        tf = tempfile.NamedTemporaryFile(suffix=".pt", delete=False)
        tf.close(); tmp = tf.name
        with zipfile.ZipFile(tmp, "w", compression=zipfile.ZIP_STORED) as zf:
            for fp in sorted(arch.rglob("*")):
                if fp.is_file():
                    info = zipfile.ZipInfo(str(fp.relative_to(arch.parent)))
                    info.date_time = (1980, 1, 1, 0, 0, 0)
                    with open(fp, "rb") as fh:
                        zf.writestr(info, fh.read())
        sd = torch.load(tmp, map_location="cpu", weights_only=False)
    else:
        log.warning("Checkpoint introuvable — poids aléatoires")
        return enc.to(device).eval()
    if tmp:
        os.unlink(tmp)
    sd = sd.get("model", sd)
    prefix = "image_encoder."
    if any(k.startswith(prefix) for k in sd):
        sd = {k[len(prefix):]: v for k, v in sd.items() if k.startswith(prefix)}
    m, u = enc.load_state_dict(sd, strict=False)
    log.info("Checkpoint : %d manquants, %d inattendus", len(m), len(u))
    return enc.to(device).eval()


def _register_hooks(enc, blocs):
    captured, handles = {}, []
    for i, block in enumerate(enc.trunk.blocks):
        key = f"block_{i}"
        if key in blocs:
            def _bh(m, inp, out, k=key):
                captured[k] = out.detach()
            handles.append(block.register_forward_hook(_bh))
    fpn_map = {0: "stage_4_fpn", 1: "stage_3_fpn", 2: "stage_2_fpn", 3: "stage_1_fpn"}
    for ci, key in fpn_map.items():
        if key in blocs:
            def _fh(m, inp, out, k=key):
                captured[k] = out.detach().permute(0, 2, 3, 1)
            handles.append(enc.neck.convs[ci].register_forward_hook(_fh))
    return captured, handles


def _preprocess(img_path, device):
    img = PILImage.open(img_path).convert("RGB")
    orig_w, orig_h = img.size
    img = img.resize((IMG_SIZE, IMG_SIZE), PILImage.BILINEAR)
    x = torch.from_numpy(np.array(img)).float() / 255.0
    x = x.permute(2, 0, 1)
    x = (x - torch.tensor(_MEAN)) / torch.tensor(_STD)
    return x.unsqueeze(0).to(device), orig_h, orig_w


def _patch_region(feat_hw, orig_h, orig_w, x_min, y_min, x_max, y_max):
    H_f, W_f = feat_hw
    sx = W_f / orig_w
    sy = H_f / orig_h
    fx1 = max(0, int(x_min * sx))
    fy1 = max(0, int(y_min * sy))
    fx2 = min(W_f, max(fx1 + 1, int(x_max * sx)))
    fy2 = min(H_f, max(fy1 + 1, int(y_max * sy)))
    return fy1, fy2, fx1, fx2


def _extract_local_vecs(feat_map, fy1, fy2, fx1, fx2):
    """Extrait TOUS les vecteurs de la région (pas de cap), L2-normalise."""
    region = feat_map[fy1:fy2, fx1:fx2, :]   # (h, w, C)
    h, w, C = region.shape
    vecs = region.reshape(-1, C).cpu().numpy().astype(np.float32)
    rr, cc = np.meshgrid(np.arange(h), np.arange(w), indexing="ij")
    rows = rr.ravel()
    cols = cc.ravel()
    norms = np.linalg.norm(vecs, axis=1, keepdims=True)
    vecs = vecs / np.maximum(norms, 1e-8)
    return vecs, rows, cols, h, w


# ── Vérification cache ────────────────────────────────────────────────
cache_vecs = {b: CACHE_DIR / f"vecs_{b}.pkl" for b in BLOCS}
cache_ok   = (not FORCE_RECOMPUTE) and all(cf.exists() for cf in cache_vecs.values())

if cache_ok:
    print("Cache vecs présent → rechargement (pas de recalcul).")
    patches_by_bloc = {}
    for b, cf in cache_vecs.items():
        with open(cf, "rb") as fh:
            patches_by_bloc[b] = pickle.load(fh)
        nv_moy = (sum(len(p["vecs"]) for p in patches_by_bloc[b]) // len(patches_by_bloc[b])
                  if patches_by_bloc[b] else 0)
        print(f"  {b} : {len(patches_by_bloc[b])} patches, ~{nv_moy} vec/patch")
else:
    print("Extraction COMPLÈTE (tous vecteurs, aucun cap).")
    print("GPU max utilisé : forward pass en FP16 (AMP) sur CUDA.\n")

    with h5py.File(H5_PATH, "r") as f:
        all_cats = f["metadata/category_ids"][:]
        all_imgs = np.array([x.decode() for x in f["metadata/image_names"][:]])
        all_pos  = f["metadata/positions"][:]

    mask = np.isin(all_cats, TEXTURES)
    cats        = all_cats[mask]
    imgs        = all_imgs[mask]
    pos         = all_pos[mask].astype(int)
    pids_global = np.where(mask)[0]
    stems       = np.array([n.replace(".tif", "") for n in imgs])
    N = int(mask.sum())
    print(f"{N} patches, {len(TEXTURES)} textures, {len(np.unique(stems))} images")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device : {device}")
    enc = _load_model(device)
    captured, handles = _register_hooks(enc, BLOCS)

    patches_by_bloc = {b: [] for b in BLOCS}
    by_img = defaultdict(list)
    for i in range(N):
        by_img[stems[i]].append(i)

    unique_stems = sorted(by_img.keys())
    for i_img, stem in enumerate(unique_stems):
        img_path = IMG_DIR / (stem + ".tif")
        if not img_path.exists():
            log.warning("[%d/%d] Image introuvable : %s.tif", i_img+1, len(unique_stems), stem)
            continue
        try:
            tensor, orig_h, orig_w = _preprocess(img_path, device)
        except Exception as e:
            log.error("[%d/%d] Prétraitement %s : %s", i_img+1, len(unique_stems), stem, e)
            continue

        captured.clear()
        with torch.no_grad():
            # AMP fp16 : forward pass 2× plus rapide sur GPU
            with torch.cuda.amp.autocast(enabled=(device == "cuda")):
                enc(tensor)

        if not all(b in captured for b in BLOCS):
            log.warning("Blocs manquants pour %s", stem)
            continue

        for i in by_img[stem]:
            x_min, y_min, x_max, y_max = pos[i]
            pid = int(pids_global[i])
            for b in BLOCS:
                feat = captured[b][0]   # (H_f, W_f, C)
                H_f, W_f, _ = feat.shape
                fy1, fy2, fx1, fx2 = _patch_region(
                    (H_f, W_f), orig_h, orig_w, x_min, y_min, x_max, y_max)
                vecs, rows, cols, fh, fw = _extract_local_vecs(feat, fy1, fy2, fx1, fx2)
                patches_by_bloc[b].append({
                    "patch_id": pid,
                    "texture":  int(cats[i]),
                    "image":    stem,
                    "vecs":     vecs,
                    "rows":     rows,
                    "cols":     cols,
                    "feat_h":   fh,
                    "feat_w":   fw,
                })

        if (i_img + 1) % 5 == 0 or i_img == len(unique_stems) - 1:
            print(f"  [{i_img+1}/{len(unique_stems)}] {stem}")

    for h in handles:
        h.remove()

    for b in BLOCS:
        cf = cache_vecs[b]
        with open(cf, "wb") as fh:
            pickle.dump(patches_by_bloc[b], fh)
        sz_mb = cf.stat().st_size / 1e6
        print(f"  Cache sauvé : {cf.name}  ({len(patches_by_bloc[b])} patches, {sz_mb:.0f} MB)")

print("\n── Résumé (tous vecteurs, sans cap) ──")
for b in BLOCS:
    pl = patches_by_bloc[b]
    if pl:
        nv_min = min(len(p["vecs"]) for p in pl)
        nv_max = max(len(p["vecs"]) for p in pl)
        nv_moy = sum(len(p["vecs"]) for p in pl) // len(pl)
        D = pl[0]["vecs"].shape[1]
        print(f"  {b:12s} : {len(pl)} patches | vec/patch min={nv_min} moy={nv_moy} max={nv_max} | dim={D}")
    else:
        print(f"  {b:12s} : 0 patches")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELLULE 4 — LP LOIO MULTICLASSE + VOTES PAR VECTEUR  (lourd — 1 fois)
# Vote sur TOUS les vecteurs de chaque patch (pas de cap).
# Protocole anti-fuite : split par IMAGE, PCA fit sur TRAIN du fold.
# Solver : saga + n_jobs=-1 (tous les CPU cores) → parallélisme max.
# ══════════════════════════════════════════════════════════════════════
diag_cache = {b: CACHE_DIR / f"diag_{b}.pkl" for b in BLOCS}
diag_ok    = (not FORCE_RECOMPUTE) and all(cf.exists() for cf in diag_cache.values())

if diag_ok:
    print("Cache diag présent → rechargement.")
    diag_by_bloc = {}
    for b, cf in diag_cache.items():
        with open(cf, "rb") as fh:
            diag_by_bloc[b] = pickle.load(fh)
        nv_moy = (sum(len(d["per_vec_voted"]) for d in diag_by_bloc[b].values())
                  // max(len(diag_by_bloc[b]), 1))
        print(f"  {b} : {len(diag_by_bloc[b])} patches, ~{nv_moy} vec/patch votés")
else:
    print("LP LOIO multiclasse — vote sur TOUS les vecteurs.\n"
          "Solver : saga, n_jobs=-1 (parallèle CPU). Peut prendre 30-90 min pour block_0.\n")
    diag_by_bloc = {}
    tex_to_i = {t: i for i, t in enumerate(TEXTURES)}

    for b in BLOCS:
        print(f"\n── Bloc : {b} ──")
        patches = patches_by_bloc[b]
        if not patches:
            diag_by_bloc[b] = {}
            continue

        all_vecs = np.concatenate([p["vecs"] for p in patches], axis=0)
        all_tex  = np.concatenate(
            [[p["texture"]] * len(p["vecs"]) for p in patches])
        all_imgs_arr = np.array(
            [p["image"] for p in patches for _ in range(len(p["vecs"]))])
        all_pids = np.concatenate(
            [[p["patch_id"]] * len(p["vecs"]) for p in patches])
        all_rows = np.concatenate([p["rows"] for p in patches])
        all_cols = np.concatenate([p["cols"] for p in patches])

        patch_hw  = {p["patch_id"]: (p["feat_h"], p["feat_w"]) for p in patches}
        patch_tex = {p["patch_id"]: p["texture"]                for p in patches}
        patch_img = {p["patch_id"]: p["image"]                  for p in patches}

        unique_imgs_lp = sorted(set(p["image"] for p in patches))
        N_total = len(all_vecs)
        print(f"  {N_total:,} vecteurs au total, {len(unique_imgs_lp)} folds LOIO")

        diag_pid = {}
        anti_fuite_shown = False

        for stem in unique_imgs_lp:
            te = all_imgs_arr == stem
            tr = ~te

            te_set = set(int(p) for p in all_pids[te])
            tr_set = set(int(p) for p in all_pids[tr])
            overlap = te_set & tr_set
            assert not overlap, f"FUITE DÉTECTÉE sur bloc {b} image {stem}: {overlap}"
            if not anti_fuite_shown:
                print(f"  Anti-fuite OK ('{stem}') : "
                      f"test={len(te_set)} patches, train={len(tr_set)} → ∅")
                anti_fuite_shown = True

            X_tr_raw = all_vecs[tr]
            X_te_raw = all_vecs[te]
            y_tr = np.array([tex_to_i[t] for t in all_tex[tr]])
            y_te = np.array([tex_to_i[t] for t in all_tex[te]])

            if len(np.unique(y_tr)) < 2:
                continue

            if X_tr_raw.shape[1] > PCA_DIM:
                pca  = PCA(n_components=PCA_DIM, random_state=SEED)
                X_tr = pca.fit_transform(X_tr_raw)
                X_te = pca.transform(X_te_raw)
            else:
                X_tr, X_te = X_tr_raw.copy(), X_te_raw.copy()

            # saga + n_jobs=-1 : utilise tous les cœurs CPU
            clf = LogisticRegression(
                C=LP_C, class_weight="balanced",
                max_iter=300, random_state=SEED,
                solver="saga", n_jobs=-1,
            )
            try:
                clf.fit(X_tr, y_tr)
            except Exception as e:
                log.warning("LP échoué (%s, %s) : %s", b, stem, e)
                continue

            known = clf.classes_
            proba_raw  = clf.predict_proba(X_te)
            full_proba = np.zeros((len(X_te), len(TEXTURES)), dtype=np.float32)
            full_proba[:, known] = proba_raw

            te_pids = all_pids[te]
            te_rows = all_rows[te]
            te_cols = all_cols[te]

            for pid in np.unique(te_pids):
                idx      = te_pids == pid
                p_proba  = full_proba[idx]
                p_rows   = te_rows[idx]
                p_cols   = te_cols[idx]

                vec_voted_idx = np.argmax(p_proba, axis=1)
                counts        = np.bincount(vec_voted_idx, minlength=len(TEXTURES))
                majority_idx  = int(np.argmax(counts))
                majority_tex  = TEXTURES[majority_idx]
                taux_accord   = int(counts[majority_idx]) / len(vec_voted_idx)

                diag_pid[int(pid)] = {
                    "texture_vraie":    patch_tex[int(pid)],
                    "vote_majoritaire": majority_tex,
                    "taux_accord":      float(taux_accord),
                    "image":            patch_img[int(pid)],
                    "per_vec_voted":    [TEXTURES[vi] for vi in vec_voted_idx.tolist()],
                    "per_vec_rows":     p_rows.tolist(),
                    "per_vec_cols":     p_cols.tolist(),
                    "feat_h":           patch_hw[int(pid)][0],
                    "feat_w":           patch_hw[int(pid)][1],
                }

        diag_by_bloc[b] = diag_pid
        nv_moy = (sum(len(d["per_vec_voted"]) for d in diag_pid.values())
                  // max(len(diag_pid), 1))
        print(f"  → {len(diag_pid)} patches diagnostiqués, ~{nv_moy} vec/patch votés")

    for b in BLOCS:
        cf = diag_cache[b]
        with open(cf, "wb") as fh:
            pickle.dump(diag_by_bloc[b], fh)
        sz_kb = cf.stat().st_size / 1e3
        print(f"Cache sauvé : {cf.name}  ({sz_kb:.0f} KB)")

# ── Contrôle anti-fuite (10 folds) ───────────────────────────────────
print("\n── Contrôle anti-fuite global (bloc 0, 10 premiers folds) ──")
b_chk = BLOCS[0]
pl_chk = patches_by_bloc[b_chk]
ai_c   = np.array([p["image"] for p in pl_chk for _ in range(len(p["vecs"]))])
ap_c   = np.concatenate([[p["patch_id"]] * len(p["vecs"]) for p in pl_chk])
leaks  = 0
for stem in sorted(set(p["image"] for p in pl_chk))[:10]:
    te_m   = ai_c == stem
    te_set = set(ap_c[te_m].astype(int))
    tr_set = set(ap_c[~te_m].astype(int))
    inter  = te_set & tr_set
    status = "✓ ∅" if not inter else f"⚠ FUITE {len(inter)} patches"
    print(f"  fold '{stem}' : test={len(te_set)}, train={len(tr_set)} → {status}")
    if inter:
        leaks += 1
print("\n✓✓ Aucune fuite détectée." if leaks == 0
      else f"\n⚠ {leaks} FUITES — résultats invalides !")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELLULE 5 — CLASSIFICATION EN 4 TYPES  (LÉGER — relancer pour tester seuils)
# Modifier SEUIL_UNANIME / SEUIL_2ZONES en cellule 1, relancer cellule 5.
# ══════════════════════════════════════════════════════════════════════

def _top2_cumul(per_vec_voted):
    """Fraction cumulée des 2 textures les plus votées."""
    counts = Counter(per_vec_voted)
    total  = len(per_vec_voted)
    if total == 0:
        return 0.0
    top2 = sorted(counts.values(), reverse=True)[:2]
    return sum(top2) / total


def _classify(taux_accord, vote_majoritaire, texture_vraie, top2):
    """
    TYPE 4 — Mal classé     : vote != étiquette
    TYPE 1 — Unanime cohérent : accord >= SEUIL_UNANIME et vote == étiquette
    TYPE 2 — Divisé en zones  : top-2 >= SEUIL_2ZONES
    TYPE 3 — Éparpillé        : reste
    """
    if vote_majoritaire != texture_vraie:
        return 4
    if taux_accord >= SEUIL_UNANIME:
        return 1
    if top2 >= SEUIL_2ZONES:
        return 2
    return 3


# Calculer les types
types_by_bloc = {}
for b in BLOCS:
    types_by_bloc[b] = {}
    for pid, d in diag_by_bloc[b].items():
        top2 = _top2_cumul(d["per_vec_voted"])
        types_by_bloc[b][pid] = _classify(
            d["taux_accord"], d["vote_majoritaire"], d["texture_vraie"], top2)

TYPE_NAMES = {1: "1-Unanime", 2: "2-Zones", 3: "3-Éparpillé", 4: "4-Mal classé"}
TYPES_LIST = [1, 2, 3, 4]

print(f"Seuils : SEUIL_UNANIME={SEUIL_UNANIME}   SEUIL_2ZONES={SEUIL_2ZONES}\n")

for b in BLOCS:
    print(f"── {STAGE_LABELS[b]} ──")
    col_w = 13
    header = f"  {'Texture':<16}" + "".join(
        f"{TYPE_NAMES[t]:>{col_w}}" for t in TYPES_LIST) + f"{'TOTAL':>8}"
    print(header)
    print("  " + "─" * (len(header) - 2))

    totals = {t: 0 for t in TYPES_LIST}
    diag   = diag_by_bloc[b]
    types  = types_by_bloc[b]

    for tex in TEXTURES:
        tex_pids = [pid for pid, d in diag.items() if d["texture_vraie"] == tex]
        counts   = {t: sum(1 for pid in tex_pids if types.get(pid) == t)
                    for t in TYPES_LIST}
        for t in TYPES_LIST:
            totals[t] += counts[t]
        row = f"  {TNAMES[tex]:<16}" + "".join(
            f"{counts[t]:>{col_w}}" if counts[t] > 0 else f"{'—':>{col_w}}"
            for t in TYPES_LIST
        ) + f"{len(tex_pids):>8}"
        print(row)

    print("  " + "─" * (len(header) - 2))
    print(f"  {'TOTAL':<16}" + "".join(
        f"{totals[t]:>{col_w}}" for t in TYPES_LIST) + "\n")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELLULE 6 — EXPLORATION INTERACTIVE  (LÉGER — modifier les variables)
# La carte spatiale montre TOUS les vecteurs (ou N_VEC_AFFICHAGE si défini).
# ══════════════════════════════════════════════════════════════════════

# ── Paramètres à modifier ────────────────────────────────────────────
TEXTURE_CHOISIE = 7          # parmi TEXTURES = [1, 3, 4, 5, 6, 7, 9]
TYPE_CHOISI     = 2          # 1=Unanime, 2=Zones, 3=Éparpillé, 4=Mal classé
BLOC_CHOISI     = "block_0"  # parmi BLOCS
N_EXEMPLES      = 3
# ─────────────────────────────────────────────────────────────────────

assert BLOC_CHOISI in BLOCS, f"BLOC_CHOISI='{BLOC_CHOISI}' invalide."
assert TEXTURE_CHOISIE in TEXTURES, f"TEXTURE_CHOISIE={TEXTURE_CHOISIE} invalide"

diag    = diag_by_bloc[BLOC_CHOISI]
types_b = types_by_bloc[BLOC_CHOISI]

selected_pids = [
    pid for pid, d in diag.items()
    if d["texture_vraie"] == TEXTURE_CHOISIE and types_b.get(pid) == TYPE_CHOISI
]
print(f"Texture={TNAMES[TEXTURE_CHOISIE]}  Type={TYPE_CHOISI}  Bloc={BLOC_CHOISI}")
print(f"→ {len(selected_pids)} patches trouvés")

if not selected_pids:
    print(f"\nAucun patch de type {TYPE_CHOISI} pour la texture '{TNAMES[TEXTURE_CHOISIE]}' "
          f"au bloc '{BLOC_CHOISI}'.")
    print("Essayez d'autres valeurs ou ajustez les seuils en cellule 1 puis relancez 5.")
else:
    reverse = TYPE_CHOISI in (1, 4)
    selected_pids.sort(key=lambda p: diag[p]["taux_accord"], reverse=reverse)
    show_pids = selected_pids[:N_EXEMPLES]

    with h5py.File(H5_PATH, "r") as f:
        all_pos_h5 = f["metadata/positions"][:]

    def _hex_to_rgb(hx):
        return [int(hx[1:3], 16)/255, int(hx[3:5], 16)/255, int(hx[5:7], 16)/255]

    def _subsample_for_display(rows, cols, voted, n_max):
        """Sous-échantillonne les vecteurs pour l'affichage uniquement."""
        if n_max is None or len(voted) <= n_max:
            return rows, cols, voted
        rng = np.random.default_rng(SEED)
        idx = rng.choice(len(voted), n_max, replace=False)
        return [rows[i] for i in idx], [cols[i] for i in idx], [voted[i] for i in idx]

    def _vote_map_rgb(fh, fw, rows, cols, voted):
        vote_map = np.full((fh, fw), -1, dtype=np.int16)
        for r, c, tv in zip(rows, cols, voted):
            if 0 <= r < fh and 0 <= c < fw:
                vote_map[r, c] = tv
        rgb = np.ones((fh, fw, 3))
        for r in range(fh):
            for c in range(fw):
                rgb[r, c] = _hex_to_rgb(TEX_COLORS.get(int(vote_map[r, c]), "#cccccc"))
        return rgb

    for pid in show_pids:
        d       = diag[pid]
        stem    = d["image"]
        t_vraie = d["texture_vraie"]
        t_vote  = d["vote_majoritaire"]
        accord  = d["taux_accord"]
        n_vecs_total = len(d["per_vec_voted"])

        # Sous-échantillonnage pour l'affichage (vote déjà calculé sur tous)
        disp_rows, disp_cols, disp_voted = _subsample_for_display(
            d["per_vec_rows"], d["per_vec_cols"], d["per_vec_voted"], N_VEC_AFFICHAGE)
        n_vecs_aff = len(disp_voted)

        img_path = IMG_DIR / (stem + ".tif")
        crop = None
        if img_path.exists():
            xmin, ymin, xmax, ymax = [int(v) for v in all_pos_h5[pid]]
            arr = np.array(PILImage.open(img_path).convert("L"))
            crop = arr[ymin:ymax, xmin:xmax]

        rgb_map = _vote_map_rgb(d["feat_h"], d["feat_w"], disp_rows, disp_cols, disp_voted)

        vote_counts = Counter(d["per_vec_voted"])  # histogramme sur TOUS les vecs
        ncols = 3 if crop is not None else 2
        fig, axes = plt.subplots(1, ncols, figsize=(5 * ncols, 4.5))

        ax_i = 0
        if crop is not None:
            axes[ax_i].imshow(crop, cmap="gray")
            axes[ax_i].set_title(f"Crop (image Ouassim)\n{stem}", fontsize=9)
            axes[ax_i].axis("off")
            ax_i += 1

        aff_label = (f"{n_vecs_aff}/{n_vecs_total} vec affichés"
                     if N_VEC_AFFICHAGE else f"{n_vecs_total} vec (tous)")
        axes[ax_i].imshow(rgb_map, aspect="auto", interpolation="nearest")
        axes[ax_i].set_title(
            f"Carte spatiale — {STAGE_LABELS[BLOC_CHOISI]}\n{aff_label}", fontsize=9)
        axes[ax_i].set_xlabel("col (feat map)"); axes[ax_i].set_ylabel("row (feat map)")
        voted_textures_set = set(d["per_vec_voted"]) | {t_vraie}
        legend_h = [
            mpatches.Patch(color=TEX_COLORS[t], label=f"t{t} {TNAMES[t][:8]}")
            for t in TEXTURES if t in voted_textures_set
        ] + [mpatches.Patch(color="#cccccc", label="non-voté")]
        axes[ax_i].legend(handles=legend_h, fontsize=7, loc="upper right", framealpha=0.85)
        ax_i += 1

        bar_tex    = sorted(TEXTURES)
        bar_vals   = [vote_counts.get(t, 0) for t in bar_tex]
        bar_colors = [TEX_COLORS[t] for t in bar_tex]
        bars = axes[ax_i].bar(range(len(bar_tex)), bar_vals,
                               color=bar_colors, edgecolor="white", linewidth=0.8)
        axes[ax_i].set_xticks(range(len(bar_tex)))
        axes[ax_i].set_xticklabels(
            [f"t{t}\n{TNAMES[t][:6]}" for t in bar_tex], fontsize=7)
        axes[ax_i].set_ylabel("Nb vecteurs votant")
        axes[ax_i].set_title(
            f"Histogramme des votes\n{n_vecs_total} vecteurs (vote complet)", fontsize=9)
        axes[ax_i].spines[["top", "right"]].set_visible(False)
        for i_b, t in enumerate(bar_tex):
            if t == t_vraie:
                bars[i_b].set_edgecolor("black")
                bars[i_b].set_linewidth(2.5)

        fig.suptitle(
            f"t{t_vraie} {TNAMES[t_vraie]} | type {TYPE_CHOISI} | accord={accord:.2f} | "
            f"annoté=t{t_vraie}  voté=t{t_vote} | {stem} | {n_vecs_total} vec. totaux",
            fontsize=9, y=1.02)
        plt.tight_layout()
        plt.show()
        print(f"patch_id={pid}  accord={accord:.2f}  "
              f"vote={TNAMES.get(t_vote,'?')}  vraitex={TNAMES.get(t_vraie,'?')}  "
              f"nVec={n_vecs_total}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELLULE 7 — SUIVI D'UN PATCH À TRAVERS LES 4 STAGES  (LÉGER)
# Carte spatiale : N_VEC_AFFICHAGE (None = tous).  Vote : toujours complet.
# ══════════════════════════════════════════════════════════════════════

# ── Variable à modifier ─────────────────────────────────────────────
PATCH_ID = (selected_pids[0]
            if 'selected_pids' in dir() and selected_pids
            else next(iter(diag_by_bloc[BLOCS[0]])))
# ────────────────────────────────────────────────────────────────────

print(f"Suivi du patch #{PATCH_ID} à travers les 4 stages")

missing_blocs = [b for b in BLOCS if PATCH_ID not in diag_by_bloc[b]]
if missing_blocs:
    print(f"⚠ patch_id={PATCH_ID} absent dans : {missing_blocs}")
    print("Essayez un autre PATCH_ID.")
    print("IDs disponibles dans block_0 :",
          sorted(diag_by_bloc[BLOCS[0]].keys())[:10], "...")
else:
    d0 = diag_by_bloc[BLOCS[0]][PATCH_ID]

    def _hex_to_rgb(hx):
        return [int(hx[1:3], 16)/255, int(hx[3:5], 16)/255, int(hx[5:7], 16)/255]

    def _subsample_for_display(rows, cols, voted, n_max):
        if n_max is None or len(voted) <= n_max:
            return rows, cols, voted
        rng = np.random.default_rng(SEED)
        idx = rng.choice(len(voted), n_max, replace=False)
        return [rows[i] for i in idx], [cols[i] for i in idx], [voted[i] for i in idx]

    def _vote_map_rgb(fh, fw, rows, cols, voted):
        vote_map = np.full((fh, fw), -1, dtype=np.int16)
        for r, c, tv in zip(rows, cols, voted):
            if 0 <= r < fh and 0 <= c < fw:
                vote_map[r, c] = tv
        rgb = np.ones((fh, fw, 3))
        for r in range(fh):
            for c in range(fw):
                rgb[r, c] = _hex_to_rgb(TEX_COLORS.get(int(vote_map[r, c]), "#cccccc"))
        return rgb

    fig, axes = plt.subplots(2, 4, figsize=(18, 8))

    for col_i, b in enumerate(BLOCS):
        d      = diag_by_bloc[b][PATCH_ID]
        t_vr   = d["texture_vraie"]
        t_vo   = d["vote_majoritaire"]
        acc    = d["taux_accord"]
        n_v    = len(d["per_vec_voted"])
        t_type = types_by_bloc[b].get(PATCH_ID, "?")

        # Sous-échantillonnage pour l'affichage uniquement
        disp_r, disp_c, disp_v = _subsample_for_display(
            d["per_vec_rows"], d["per_vec_cols"], d["per_vec_voted"], N_VEC_AFFICHAGE)
        n_aff = len(disp_v)
        rgb = _vote_map_rgb(d["feat_h"], d["feat_w"], disp_r, disp_c, disp_v)

        aff_label = (f"{n_aff}/{n_v} aff." if N_VEC_AFFICHAGE else f"{n_v} vec. (tous)")

        ax_s = axes[0, col_i]
        ax_s.imshow(rgb, aspect="auto", interpolation="nearest")
        label_vote = (f"annoté=voté=t{t_vr}" if t_vo == t_vr
                      else f"annoté=t{t_vr} voté=t{t_vo}")
        ax_s.set_title(
            f"{STAGE_LABELS[b]}\nacc={acc:.2f}  {aff_label}  type={t_type}\n{label_vote}",
            fontsize=8)
        ax_s.axis("off")

        if col_i == 0:
            leg = [
                mpatches.Patch(color=TEX_COLORS[t], label=f"t{t} {TNAMES[t][:7]}")
                for t in TEXTURES
            ] + [mpatches.Patch(color="#cccccc", label="non-voté")]
            ax_s.legend(handles=leg, fontsize=6, loc="lower left", framealpha=0.8)

        # Histogramme sur TOUS les vecteurs
        vote_counts = Counter(d["per_vec_voted"])
        ax_h = axes[1, col_i]
        bar_tex  = sorted(TEXTURES)
        bar_vals = [vote_counts.get(t, 0) for t in bar_tex]
        bars = ax_h.bar(range(len(bar_tex)), bar_vals,
                        color=[TEX_COLORS[t] for t in bar_tex],
                        edgecolor="white", linewidth=0.8)
        ax_h.set_xticks(range(len(bar_tex)))
        ax_h.set_xticklabels([f"t{t}" for t in bar_tex], fontsize=7)
        ax_h.set_ylabel("Nb vecteurs", fontsize=8)
        ax_h.set_title(f"{n_v} vec. totaux", fontsize=8)
        ax_h.spines[["top", "right"]].set_visible(False)
        for i_b, t in enumerate(bar_tex):
            if t == t_vr:
                bars[i_b].set_edgecolor("black")
                bars[i_b].set_linewidth(2.5)

    d15  = diag_by_bloc["block_15"][PATCH_ID]
    n15  = len(d15.get("per_vec_voted", []))
    note = (f"⚠ Stage 4 (block_15) : seulement {n15} vecteurs → carte grossière.\n"
            ) if n15 < 20 else ""

    fig.suptitle(
        f"Patch #{PATCH_ID} — texture {TNAMES.get(d0['texture_vraie'], '?')} — "
        f"évolution sur les 4 stages\n{note}",
        fontsize=10, y=1.02)
    plt.tight_layout()
    plt.show()

    print(f"\nRésumé patch #{PATCH_ID} (texture={TNAMES.get(d0['texture_vraie'], '?')}) :")
    for b in BLOCS:
        d = diag_by_bloc[b][PATCH_ID]
        print(f"  {b:12s} : accord={d['taux_accord']:.2f}  "
              f"vote={TNAMES.get(d['vote_majoritaire'], '?'):<14}  "
              f"nVec={len(d['per_vec_voted']):4d}  "
              f"type={types_by_bloc[b].get(PATCH_ID, '?')}")